<a href="https://colab.research.google.com/github/christo444/Dark-Guard/blob/main/Dark_guard_models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import sys
import sklearn
import pandas as pd
import numpy as np

print("Python:", sys.version)
print("scikit-learn:", sklearn.__version__)
print("pandas:", pd.__version__)
print("numpy:", np.__version__)


Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
scikit-learn: 1.6.1
pandas: 2.2.3
numpy: 2.1.3


In [2]:
DATA_URL = "https://raw.githubusercontent.com/yamanalab/ec-darkpattern/master/dataset/dataset.tsv"
df = pd.read_csv(DATA_URL, sep="\t")

print("Shape:", df.shape)                       # expect (2356, 4)
print("Columns:", list(df.columns))              # page_id, text, label, Pattern Category
print("Missing values:\n", df.isnull().sum())    # expect all zero
print("Duplicate rows:", df.duplicated().sum())  # expect 0
print("Label counts:\n", df["label"].value_counts())  # expect 1178 / 1178



Shape: (2356, 4)
Columns: ['page_id', 'text', 'label', 'Pattern Category']
Missing values:
 page_id             0
text                0
label               0
Pattern Category    0
dtype: int64
Duplicate rows: 0
Label counts:
 label
1    1178
0    1178
Name: count, dtype: int64


In [3]:
from sklearn.model_selection import StratifiedGroupKFold

outer = StratifiedGroupKFold(n_splits=7, shuffle=True, random_state=42)
trainval_idx, test_idx = next(
    outer.split(df["text"], df["label"], groups=df["page_id"])
)
trainval_df = df.iloc[trainval_idx].reset_index(drop=True)
test_df = df.iloc[test_idx].reset_index(drop=True)

overlap = set(trainval_df.page_id) & set(test_df.page_id)
print(f"Train+val pool: {len(trainval_df)} rows")
print(f"Held-out test:  {len(test_df)} rows")
print(f"page_id overlap (must be 0): {len(overlap)}")
# expect: 2019 / 337 / 0


Train+val pool: 2072 rows
Held-out test:  284 rows
page_id overlap (must be 0): 0


In [4]:
inner = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
cv_splits = list(
    inner.split(trainval_df["text"], trainval_df["label"], groups=trainval_df["page_id"])
)

for fold, (tr_idx, val_idx) in enumerate(cv_splits, start=1):
    tr, val = trainval_df.iloc[tr_idx], trainval_df.iloc[val_idx]
    fold_overlap = set(tr.page_id) & set(val.page_id)
    print(f"Fold {fold}: train={len(tr)} val={len(val)} "
          f"val_labels={val['label'].value_counts().to_dict()} "
          f"page_id_overlap={len(fold_overlap)}")

Fold 1: train=1693 val=379 val_labels={1: 200, 0: 179} page_id_overlap=0
Fold 2: train=1691 val=381 val_labels={1: 206, 0: 175} page_id_overlap=0
Fold 3: train=1674 val=398 val_labels={1: 203, 0: 195} page_id_overlap=0
Fold 4: train=1511 val=561 val_labels={0: 366, 1: 195} page_id_overlap=0
Fold 5: train=1719 val=353 val_labels={1: 207, 0: 146} page_id_overlap=0


In [5]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

rf_rows = []
for fold, (tr_idx, val_idx) in enumerate(cv_splits, start=1):
    tr, val = trainval_df.iloc[tr_idx], trainval_df.iloc[val_idx]

    vectorizer = CountVectorizer()
    X_tr = vectorizer.fit_transform(tr["text"])
    X_val = vectorizer.transform(val["text"])

    clf = RandomForestClassifier(
        n_estimators=641, max_depth=32, min_samples_split=18,
        min_samples_leaf=1, bootstrap=False, random_state=42, n_jobs=-1
    )
    clf.fit(X_tr, tr["label"])
    preds = clf.predict(X_val)

    acc = accuracy_score(val["label"], preds)
    prec = precision_score(val["label"], preds)
    rec = recall_score(val["label"], preds)
    f1 = f1_score(val["label"], preds)
    rf_rows.append([fold, acc, prec, rec, f1])
    print(f"Fold {fold}: acc={acc:.4f} prec={prec:.4f} rec={rec:.4f} f1={f1:.4f}")

rf_arr = np.array([r[1:] for r in rf_rows])
print()
print(f"RF Mean:  acc={rf_arr[:,0].mean():.4f} prec={rf_arr[:,1].mean():.4f} "
      f"rec={rf_arr[:,2].mean():.4f} f1={rf_arr[:,3].mean():.4f}")
print(f"RF Std:   acc={rf_arr[:,0].std():.4f} prec={rf_arr[:,1].std():.4f} "
      f"rec={rf_arr[:,2].std():.4f} f1={rf_arr[:,3].std():.4f}")

Fold 1: acc=0.9578 prec=0.9842 rec=0.9350 f1=0.9590
Fold 2: acc=0.9370 prec=0.9840 rec=0.8981 f1=0.9391
Fold 3: acc=0.9422 prec=0.9500 rec=0.9360 f1=0.9429
Fold 4: acc=0.9554 prec=0.9293 rec=0.9436 f1=0.9364
Fold 5: acc=0.9235 prec=0.9639 rec=0.9034 f1=0.9327

RF Mean:  acc=0.9432 prec=0.9623 rec=0.9232 f1=0.9420
RF Std:   acc=0.0126 prec=0.0210 rec=0.0187 f1=0.0091


In [6]:
from sklearn.svm import SVC

svm_rows = []
for fold, (tr_idx, val_idx) in enumerate(cv_splits, start=1):
    tr, val = trainval_df.iloc[tr_idx], trainval_df.iloc[val_idx]

    vectorizer = CountVectorizer()
    X_tr = vectorizer.fit_transform(tr["text"])
    X_val = vectorizer.transform(val["text"])

    clf = SVC(kernel="rbf", C=4.35, random_state=42)
    clf.fit(X_tr, tr["label"])
    preds = clf.predict(X_val)

    acc = accuracy_score(val["label"], preds)
    prec = precision_score(val["label"], preds)
    rec = recall_score(val["label"], preds)
    f1 = f1_score(val["label"], preds)
    svm_rows.append([fold, acc, prec, rec, f1])
    print(f"Fold {fold}: acc={acc:.4f} prec={prec:.4f} rec={rec:.4f} f1={f1:.4f}")

svm_arr = np.array([r[1:] for r in svm_rows])
print()
print(f"SVM Mean: acc={svm_arr[:,0].mean():.4f} prec={svm_arr[:,1].mean():.4f} "
      f"rec={svm_arr[:,2].mean():.4f} f1={svm_arr[:,3].mean():.4f}")
print(f"SVM Std:  acc={svm_arr[:,0].std():.4f} prec={svm_arr[:,1].std():.4f} "
      f"rec={svm_arr[:,2].std():.4f} f1={svm_arr[:,3].std():.4f}")

Fold 1: acc=0.9551 prec=0.9692 rec=0.9450 f1=0.9570
Fold 2: acc=0.9370 prec=0.9643 rec=0.9175 f1=0.9403
Fold 3: acc=0.9271 prec=0.9223 rec=0.9360 f1=0.9291
Fold 4: acc=0.9537 prec=0.9333 rec=0.9333 f1=0.9333
Fold 5: acc=0.9405 prec=0.9895 rec=0.9082 f1=0.9471

SVM Mean: acc=0.9427 prec=0.9557 rec=0.9280 f1=0.9414
SVM Std:  acc=0.0105 prec=0.0245 rec=0.0133 f1=0.0099


In [7]:
summary = pd.DataFrame({
    "Model": ["Random Forest", "SVM"],
    "Accuracy": [rf_arr[:,0].mean(), svm_arr[:,0].mean()],
    "Precision": [rf_arr[:,1].mean(), svm_arr[:,1].mean()],
    "Recall": [rf_arr[:,2].mean(), svm_arr[:,2].mean()],
    "F1": [rf_arr[:,3].mean(), svm_arr[:,3].mean()],
    "Std (Accuracy)": [rf_arr[:,0].std(), svm_arr[:,0].std()],
})
print(summary.to_string(index=False))

        Model  Accuracy  Precision   Recall       F1  Std (Accuracy)
Random Forest  0.943190   0.962293 0.923198 0.942009        0.012564
          SVM  0.942691   0.955731 0.927996 0.941358        0.010529


In [8]:
import pandas as pd
df = pd.read_csv("darkguard_split_v1.tsv", sep="\t")
print(df["split"].value_counts())

split
trainval    2072
test         284
Name: count, dtype: int64


In [9]:
!pip install -q transformers

from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained("bert-base-uncased")

text = "FLASH SALE | LIMITED TIME ONLY Shop Now"
print("Original text:", repr(text))
print()

# Step 1: split into raw tokens (no numbers yet)
tokens = tok.tokenize(text)
print("Step 1 - Tokens:", tokens)
print()

# Step 2: convert to numeric IDs + add BERT's special tokens ([CLS], [SEP])
encoded = tok(text, padding="max_length", max_length=16, truncation=True)
print("Step 2 - input_ids:", encoded["input_ids"])
print("Step 3 - attention_mask:", encoded["attention_mask"])
print()

# Step 4: show id <-> token side by side
for tid, mask in zip(encoded["input_ids"], encoded["attention_mask"]):
    piece = tok.convert_ids_to_tokens([tid])[0]
    print(f"  id={tid:>6}  token={piece!r:<15}  attention_mask={mask}")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Original text: 'FLASH SALE | LIMITED TIME ONLY Shop Now'

Step 1 - Tokens: ['flash', 'sale', '|', 'limited', 'time', 'only', 'shop', 'now']

Step 2 - input_ids: [101, 5956, 5096, 1064, 3132, 2051, 2069, 4497, 2085, 102, 0, 0, 0, 0, 0, 0]
Step 3 - attention_mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0]

  id=   101  token='[CLS]'          attention_mask=1
  id=  5956  token='flash'          attention_mask=1
  id=  5096  token='sale'           attention_mask=1
  id=  1064  token='|'              attention_mask=1
  id=  3132  token='limited'        attention_mask=1
  id=  2051  token='time'           attention_mask=1
  id=  2069  token='only'           attention_mask=1
  id=  4497  token='shop'           attention_mask=1
  id=  2085  token='now'            attention_mask=1
  id=   102  token='[SEP]'          attention_mask=1
  id=     0  token='[PAD]'          attention_mask=0
  id=     0  token='[PAD]'          attention_mask=0
  id=     0  token='[PAD]'          attention_mask=

In [10]:
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained("bert-base-uncased")

text = ("CANCELLATION POLICY: If you are not completely satisfied with Yogi Surprise, "
        "you can cancel your membership and discontinue your monthly payments at any time. "
        "In order to cancel before you are renewed for the next shipment, you need to contact "
        "Yogi Surprise prior to the renewal date on the 14th of each month. Email: "
        "support@yogisurprise.com and we\u2019ll respond promptly. If your membership has already "
        "renewed and you\u2019d like to cancel an upcoming shipment, you must request cancellation "
        "by the 3rd of the month to be eligible for a refund. IF YOU DO NOT CANCEL PRIOR TO THE "
        "THIRD DAY OF A CALENDAR MONTH, YOU CANNOT BE REFUNDED FOR THAT MONTH\u2019S SHIPMENT. "
        "WE DO NOT OFFER REFUNDS ON BOXES THAT HAVE ALREADY BEEN SHIPPED. To request cancellation, "
        "simply click here and submit the contact form or email us: support@yogisurprise.com. "
        "We\u2019ll get back to promptly!")

# 1. How many tokens does this text actually need, with NO truncation?
full_tokens = tok.tokenize(text)
print("Total tokens needed (no truncation):", len(full_tokens))
print()

# 2. Show where the brand name / email address get split
interesting = [t for t in full_tokens if "yogi" in t.lower() or "surprise" in t.lower() or "yogisurprise" in t.lower() or "@" in t or "##" in t]
print("Tokens touching the brand name / email (watch for ## pieces):")
print(interesting[:25])
print()

# 3. Compare what survives at three different max_length settings
for max_len in [32, 64, 128]:
    encoded = tok(text, padding="max_length", max_length=max_len, truncation=True)
    kept = sum(encoded["attention_mask"])
    decoded_kept = tok.decode(encoded["input_ids"], skip_special_tokens=True)
    print(f"--- max_length={max_len} ---")
    print(f"Real tokens kept: {kept - 2} of {len(full_tokens)}")  # -2 for CLS/SEP
    print(f"What survives: {decoded_kept[:200]}...")
    print()

Total tokens needed (no truncation): 192

Tokens touching the brand name / email (watch for ## pieces):
['##gi', 'surprise', '##nti', '##nu', '##e', '##gi', 'surprise', '@', '##gis', '##ur', '##pr', '##ise', '##und', '##und', '##ed', '##unds', '@', '##gis', '##ur', '##pr', '##ise']

--- max_length=32 ---
Real tokens kept: 30 of 192
What survives: cancellation policy : if you are not completely satisfied with yogi surprise, you can cancel your membership and discontinue your monthly payments at any time...

--- max_length=64 ---
Real tokens kept: 62 of 192
What survives: cancellation policy : if you are not completely satisfied with yogi surprise, you can cancel your membership and discontinue your monthly payments at any time. in order to cancel before you are renewe...

--- max_length=128 ---
Real tokens kept: 126 of 192
What survives: cancellation policy : if you are not completely satisfied with yogi surprise, you can cancel your membership and discontinue your monthly payments at a

In [11]:
from transformers import AutoTokenizer
import pandas as pd

tok = AutoTokenizer.from_pretrained("bert-base-uncased")
df = pd.read_csv("darkguard_split_v1.tsv", sep="\t")
trainval = df[df["split"] == "trainval"]

lengths = trainval["text"].apply(lambda t: len(tok.tokenize(str(t))))

print("Token length distribution across the training pool:")
print(lengths.describe())
print()
for p in [50, 75, 90, 95, 99, 100]:
    print(f"  {p}th percentile: {lengths.quantile(p/100):.0f} tokens")

Token length distribution across the training pool:
count    2072.000000
mean        9.847490
std        14.766193
min         0.000000
25%         4.000000
50%         6.000000
75%        10.000000
max       192.000000
Name: text, dtype: float64

  50th percentile: 6 tokens
  75th percentile: 10 tokens
  90th percentile: 19 tokens
  95th percentile: 27 tokens
  99th percentile: 82 tokens
  100th percentile: 192 tokens


In [12]:
from transformers import AutoTokenizer
import pandas as pd

tok = AutoTokenizer.from_pretrained("bert-base-uncased")
df = pd.read_csv("darkguard_split_v1.tsv", sep="\t")
trainval = df[df["split"] == "trainval"]

lengths = trainval["text"].apply(lambda t: len(tok.tokenize(str(t))))

print("Token length distribution across the training pool:")
print(lengths.describe())
print()
for p in [50, 75, 90, 95, 99, 100]:
    print(f"  {p}th percentile: {lengths.quantile(p/100):.0f} tokens")

Token length distribution across the training pool:
count    2072.000000
mean        9.847490
std        14.766193
min         0.000000
25%         4.000000
50%         6.000000
75%        10.000000
max       192.000000
Name: text, dtype: float64

  50th percentile: 6 tokens
  75th percentile: 10 tokens
  90th percentile: 19 tokens
  95th percentile: 27 tokens
  99th percentile: 82 tokens
  100th percentile: 192 tokens


In [13]:
# Part A: find the empty/near-empty rows
empty_rows = trainval[trainval["text"].apply(lambda t: len(tok.tokenize(str(t))) == 0)]
print("Rows with 0 tokens:")
print(empty_rows[["page_id", "text", "label", "Pattern Category"]])
print()

# Part B: exact truncation counts at candidate max_length values
for max_len in [16, 32, 64, 128]:
    # -2 reserves room for [CLS] and [SEP]
    truncated = (lengths > (max_len - 2)).sum()
    pct = truncated / len(lengths) * 100
    print(f"max_length={max_len}: {truncated} of {len(lengths)} rows truncated ({pct:.1f}%)")

Rows with 0 tokens:
      page_id text  label  Pattern Category
1099      568          0  Not Dark Pattern

max_length=16: 307 of 2072 rows truncated (14.8%)
max_length=32: 88 of 2072 rows truncated (4.2%)
max_length=64: 27 of 2072 rows truncated (1.3%)
max_length=128: 9 of 2072 rows truncated (0.4%)


In [14]:
import pandas as pd
from transformers import AutoTokenizer

MAX_LENGTH = 64

tok = AutoTokenizer.from_pretrained("bert-base-uncased")
df = pd.read_csv("darkguard_split_v1.tsv", sep="\t")

# Section 4 cleaning decision: drop the one icon-font artifact row we found
def has_pua(text):
    return any(0xE000 <= ord(c) <= 0xF8FF for c in str(text))

before = len(df)
df = df[~df["text"].apply(has_pua)].reset_index(drop=True)
print(f"Dropped {before - len(df)} row(s). New shape: {df.shape}")

# Encode every remaining row in one call
encodings = tok(
    list(df["text"]),
    padding="max_length",
    truncation=True,
    max_length=MAX_LENGTH,
)

print("Number of encoded rows:", len(encodings["input_ids"]))
print("Length of input_ids for row 0:", len(encodings["input_ids"][0]))
print("Length of attention_mask for row 0:", len(encodings["attention_mask"][0]))
print()
print("Row 0 text:", df.iloc[0]["text"])
print("Row 0 label:", df.iloc[0]["label"])
print("Row 0 input_ids:", encodings["input_ids"][0])

Dropped 1 row(s). New shape: (2355, 6)
Number of encoded rows: 2355
Length of input_ids for row 0: 64
Length of attention_mask for row 0: 64

Row 0 text: FLASH SALE | LIMITED TIME ONLY Shop Now
Row 0 label: 1
Row 0 input_ids: [101, 5956, 5096, 1064, 3132, 2051, 2069, 4497, 2085, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


In [15]:
import torch

# 1. GPU check
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

# 2. Wrap encodings + labels into a Dataset
class DarkPatternDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

full_dataset = DarkPatternDataset(encodings, list(df["label"]))

# sanity check on one item
sample = full_dataset[0]
print()
print("Sample keys:", sample.keys())
print("input_ids shape:", sample["input_ids"].shape)
print("attention_mask shape:", sample["attention_mask"].shape)
print("label:", sample["labels"].item())

CUDA available: True
GPU name: Tesla T4

Sample keys: dict_keys(['input_ids', 'token_type_ids', 'attention_mask', 'labels'])
input_ids shape: torch.Size([64])
attention_mask shape: torch.Size([64])
label: 1


In [16]:
import numpy as np
from torch.utils.data import Subset
from transformers import BertForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

FOLD = 1

train_idx = df.index[(df["split"] == "trainval") & (df["cv_fold"] != FOLD)].tolist()
val_idx   = df.index[(df["split"] == "trainval") & (df["cv_fold"] == FOLD)].tolist()

train_dataset = Subset(full_dataset, train_idx)
val_dataset   = Subset(full_dataset, val_idx)

print(f"Fold {FOLD}: train={len(train_dataset)}  val={len(val_dataset)}")

model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "precision": precision_score(labels, preds),
        "recall": recall_score(labels, preds),
        "f1": f1_score(labels, preds),
    }

training_args = TrainingArguments(
    output_dir="./bert_fold1_output",
    num_train_epochs=4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    seed=42,
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=20,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

trainer.train()
final_metrics = trainer.evaluate()
print(final_metrics)

Fold 1: train=1693  val=378


model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.101772,0.161886,0.949735,0.924883,0.985000,0.953995
2,0.104170,0.125623,0.957672,0.938095,0.985000,0.960976
3,0.069248,0.110640,0.973545,0.965686,0.985000,0.975248
4,0.011894,0.118895,0.965608,0.951691,0.985000,0.968059


Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.011894,0.118895,4,0.965608,0.951691,0.985000,0.968059


{'eval_loss': 0.11889512091875076, 'eval_accuracy': 0.9656084656084656, 'eval_precision': 0.9516908212560387, 'eval_recall': 0.985, 'eval_f1': 0.9680589680589681}


In [17]:
import numpy as np

all_fold_metrics = []

for FOLD in range(1, 6):
    train_idx = df.index[(df["split"] == "trainval") & (df["cv_fold"] != FOLD)].tolist()
    val_idx   = df.index[(df["split"] == "trainval") & (df["cv_fold"] == FOLD)].tolist()

    train_dataset = Subset(full_dataset, train_idx)
    val_dataset   = Subset(full_dataset, val_idx)

    model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)

    training_args = TrainingArguments(
        output_dir=f"./bert_fold{FOLD}_output",
        num_train_epochs=4,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        learning_rate=2e-5,
        seed=42,
        eval_strategy="epoch",
        save_strategy="no",
        logging_steps=50,
        report_to="none",
        disable_tqdm=True,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=compute_metrics,
    )

    trainer.train()
    metrics = trainer.evaluate()
    metrics["fold"] = FOLD
    all_fold_metrics.append(metrics)
    print(f"Fold {FOLD} done: acc={metrics['eval_accuracy']:.4f} f1={metrics['eval_f1']:.4f}")

acc = [m["eval_accuracy"] for m in all_fold_metrics]
prec = [m["eval_precision"] for m in all_fold_metrics]
rec = [m["eval_recall"] for m in all_fold_metrics]
f1 = [m["eval_f1"] for m in all_fold_metrics]

print()
print(f"Mean accuracy:  {np.mean(acc):.4f}  (std {np.std(acc):.4f})")
print(f"Mean precision: {np.mean(prec):.4f}  (std {np.std(prec):.4f})")
print(f"Mean recall:    {np.mean(rec):.4f}  (std {np.std(rec):.4f})")
print(f"Mean f1:        {np.mean(f1):.4f}  (std {np.std(f1):.4f})")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


{'loss': '0.3558', 'grad_norm': '6.399', 'learning_rate': '1.769e-05', 'epoch': '0.4717'}
{'loss': '0.1406', 'grad_norm': '25.53', 'learning_rate': '1.533e-05', 'epoch': '0.9434'}
{'eval_loss': '0.1573', 'eval_accuracy': '0.9471', 'eval_precision': '0.9327', 'eval_recall': '0.97', 'eval_f1': '0.951', 'eval_runtime': '1.433', 'eval_samples_per_second': '263.8', 'eval_steps_per_second': '16.75', 'epoch': '1'}
{'loss': '0.09641', 'grad_norm': '2.014', 'learning_rate': '1.297e-05', 'epoch': '1.415'}
{'loss': '0.09305', 'grad_norm': '4.941', 'learning_rate': '1.061e-05', 'epoch': '1.887'}
{'eval_loss': '0.106', 'eval_accuracy': '0.9683', 'eval_precision': '0.9747', 'eval_recall': '0.965', 'eval_f1': '0.9698', 'eval_runtime': '1.498', 'eval_samples_per_second': '252.3', 'eval_steps_per_second': '16.02', 'epoch': '2'}
{'loss': '0.04437', 'grad_norm': '0.0993', 'learning_rate': '8.255e-06', 'epoch': '2.358'}
{'loss': '0.0324', 'grad_norm': '6.699', 'learning_rate': '5.896e-06', 'epoch': '2.83'

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


{'loss': '0.4068', 'grad_norm': '10.62', 'learning_rate': '1.742e-05', 'epoch': '0.5263'}
{'eval_loss': '0.1315', 'eval_accuracy': '0.9626', 'eval_precision': '0.9394', 'eval_recall': '0.9538', 'eval_f1': '0.9466', 'eval_runtime': '2.196', 'eval_samples_per_second': '255.4', 'eval_steps_per_second': '16.39', 'epoch': '1'}
{'loss': '0.1474', 'grad_norm': '1.082', 'learning_rate': '1.479e-05', 'epoch': '1.053'}
{'loss': '0.08339', 'grad_norm': '2.725', 'learning_rate': '1.216e-05', 'epoch': '1.579'}
{'eval_loss': '0.1113', 'eval_accuracy': '0.9715', 'eval_precision': '0.9686', 'eval_recall': '0.9487', 'eval_f1': '0.9585', 'eval_runtime': '2.15', 'eval_samples_per_second': '260.9', 'eval_steps_per_second': '16.75', 'epoch': '2'}
{'loss': '0.06663', 'grad_norm': '0.124', 'learning_rate': '9.526e-06', 'epoch': '2.105'}
{'loss': '0.01582', 'grad_norm': '0.1892', 'learning_rate': '6.895e-06', 'epoch': '2.632'}
{'eval_loss': '0.1215', 'eval_accuracy': '0.9715', 'eval_precision': '0.9686', 'eva

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


{'loss': '0.3695', 'grad_norm': '9.517', 'learning_rate': '1.773e-05', 'epoch': '0.463'}
{'loss': '0.157', 'grad_norm': '19.35', 'learning_rate': '1.542e-05', 'epoch': '0.9259'}
{'eval_loss': '0.1492', 'eval_accuracy': '0.9462', 'eval_precision': '0.9393', 'eval_recall': '0.971', 'eval_f1': '0.9549', 'eval_runtime': '1.363', 'eval_samples_per_second': '259', 'eval_steps_per_second': '16.88', 'epoch': '1'}
{'loss': '0.1197', 'grad_norm': '1.26', 'learning_rate': '1.31e-05', 'epoch': '1.389'}
{'loss': '0.04601', 'grad_norm': '1.561', 'learning_rate': '1.079e-05', 'epoch': '1.852'}
{'eval_loss': '0.1388', 'eval_accuracy': '0.9717', 'eval_precision': '0.9805', 'eval_recall': '0.971', 'eval_f1': '0.9757', 'eval_runtime': '1.359', 'eval_samples_per_second': '259.7', 'eval_steps_per_second': '16.92', 'epoch': '2'}
{'loss': '0.04234', 'grad_norm': '0.1091', 'learning_rate': '8.472e-06', 'epoch': '2.315'}
{'loss': '0.0317', 'grad_norm': '0.08713', 'learning_rate': '6.157e-06', 'epoch': '2.778'}

In [18]:
import time
from transformers import AutoTokenizer, DistilBertForSequenceClassification

# Re-tokenize with DistilBERT's own tokenizer (verify, don't assume, it matches BERT's)
distil_tok = AutoTokenizer.from_pretrained("distilbert-base-uncased")

distil_encodings = distil_tok(
    list(df["text"]),
    padding="max_length",
    truncation=True,
    max_length=MAX_LENGTH,
)

# Quick check: does it tokenize identically to BERT's tokenizer on row 0?
print("BERT input_ids[0]:      ", encodings["input_ids"][0])
print("DistilBERT input_ids[0]:", distil_encodings["input_ids"][0])
print("Identical:", encodings["input_ids"][0] == distil_encodings["input_ids"][0])
print()

distil_dataset = DarkPatternDataset(distil_encodings, list(df["label"]))

distil_fold_metrics = []

for FOLD in range(1, 6):
    train_idx = df.index[(df["split"] == "trainval") & (df["cv_fold"] != FOLD)].tolist()
    val_idx   = df.index[(df["split"] == "trainval") & (df["cv_fold"] == FOLD)].tolist()

    train_dataset = Subset(distil_dataset, train_idx)
    val_dataset   = Subset(distil_dataset, val_idx)

    model = DistilBertForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)

    training_args = TrainingArguments(
        output_dir=f"./distilbert_fold{FOLD}_output",
        num_train_epochs=4,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        learning_rate=2e-5,
        seed=42,
        eval_strategy="epoch",
        save_strategy="no",
        logging_steps=50,
        report_to="none",
        disable_tqdm=True,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=compute_metrics,
    )

    start = time.time()
    trainer.train()
    elapsed = time.time() - start

    metrics = trainer.evaluate()
    metrics["fold"] = FOLD
    metrics["train_time_sec"] = elapsed
    distil_fold_metrics.append(metrics)
    print(f"Fold {FOLD} done: acc={metrics['eval_accuracy']:.4f} f1={metrics['eval_f1']:.4f} time={elapsed:.1f}s")

acc = [m["eval_accuracy"] for m in distil_fold_metrics]
f1 = [m["eval_f1"] for m in distil_fold_metrics]
times = [m["train_time_sec"] for m in distil_fold_metrics]

print()
print(f"DistilBERT Mean accuracy: {np.mean(acc):.4f} (std {np.std(acc):.4f})")
print(f"DistilBERT Mean f1:       {np.mean(f1):.4f} (std {np.std(f1):.4f})")
print(f"DistilBERT Mean train time per fold: {np.mean(times):.1f}s")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

BERT input_ids[0]:       [101, 5956, 5096, 1064, 3132, 2051, 2069, 4497, 2085, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
DistilBERT input_ids[0]: [101, 5956, 5096, 1064, 3132, 2051, 2069, 4497, 2085, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Identical: True



...

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


{'loss': '0.3634', 'grad_norm': '3.642', 'learning_rate': '1.742e-05', 'epoch': '0.5263'}
{'eval_loss': '0.1292', 'eval_accuracy': '0.9554', 'eval_precision': '0.9208', 'eval_recall': '0.9538', 'eval_f1': '0.937', 'eval_runtime': '1.085', 'eval_samples_per_second': '517.2', 'eval_steps_per_second': '33.19', 'epoch': '1'}
{'loss': '0.1541', 'grad_norm': '0.4368', 'learning_rate': '1.479e-05', 'epoch': '1.053'}
{'loss': '0.08826', 'grad_norm': '1.963', 'learning_rate': '1.216e-05', 'epoch': '1.579'}
{'eval_loss': '0.1149', 'eval_accuracy': '0.9697', 'eval_precision': '0.9635', 'eval_recall': '0.9487', 'eval_f1': '0.9561', 'eval_runtime': '1.092', 'eval_samples_per_second': '513.9', 'eval_steps_per_second': '32.98', 'epoch': '2'}
{'loss': '0.07847', 'grad_norm': '0.8006', 'learning_rate': '9.526e-06', 'epoch': '2.105'}
{'loss': '0.02966', 'grad_norm': '22.43', 'learning_rate': '6.895e-06', 'epoch': '2.632'}
{'eval_loss': '0.1261', 'eval_accuracy': '0.9697', 'eval_precision': '0.9785', 'ev

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


{'loss': '0.3731', 'grad_norm': '9.073', 'learning_rate': '1.773e-05', 'epoch': '0.463'}
{'loss': '0.1527', 'grad_norm': '3.391', 'learning_rate': '1.542e-05', 'epoch': '0.9259'}
{'eval_loss': '0.1494', 'eval_accuracy': '0.9462', 'eval_precision': '0.9434', 'eval_recall': '0.9662', 'eval_f1': '0.9547', 'eval_runtime': '0.6834', 'eval_samples_per_second': '516.5', 'eval_steps_per_second': '33.66', 'epoch': '1'}
{'loss': '0.1407', 'grad_norm': '0.4374', 'learning_rate': '1.31e-05', 'epoch': '1.389'}
{'loss': '0.06282', 'grad_norm': '1.277', 'learning_rate': '1.079e-05', 'epoch': '1.852'}
{'eval_loss': '0.1471', 'eval_accuracy': '0.9575', 'eval_precision': '0.9571', 'eval_recall': '0.971', 'eval_f1': '0.964', 'eval_runtime': '0.6847', 'eval_samples_per_second': '515.6', 'eval_steps_per_second': '33.59', 'epoch': '2'}
{'loss': '0.06028', 'grad_norm': '0.1001', 'learning_rate': '8.472e-06', 'epoch': '2.315'}
{'loss': '0.05326', 'grad_norm': '0.1394', 'learning_rate': '6.157e-06', 'epoch': '

In [19]:
from transformers import AutoTokenizer, RobertaForSequenceClassification

roberta_tok = AutoTokenizer.from_pretrained("roberta-base")

roberta_encodings = roberta_tok(
    list(df["text"]),
    padding="max_length",
    truncation=True,
    max_length=MAX_LENGTH,
)

# Sanity check: RoBERTa uses a DIFFERENT tokenizer/vocabulary than BERT
# (byte-level BPE, not WordPiece) - so we expect these to differ, unlike
# the BERT/DistilBERT comparison which matched exactly.
print("BERT input_ids[0]:    ", encodings["input_ids"][0])
print("RoBERTa input_ids[0]: ", roberta_encodings["input_ids"][0])
print("Identical:", encodings["input_ids"][0] == roberta_encodings["input_ids"][0])
print()

roberta_dataset = DarkPatternDataset(roberta_encodings, list(df["label"]))

roberta_fold_metrics = []

for FOLD in range(1, 6):
    train_idx = df.index[(df["split"] == "trainval") & (df["cv_fold"] != FOLD)].tolist()
    val_idx   = df.index[(df["split"] == "trainval") & (df["cv_fold"] == FOLD)].tolist()

    train_dataset = Subset(roberta_dataset, train_idx)
    val_dataset   = Subset(roberta_dataset, val_idx)

    model = RobertaForSequenceClassification.from_pretrained("roberta-base", num_labels=2)

    training_args = TrainingArguments(
        output_dir=f"./roberta_fold{FOLD}_output",
        num_train_epochs=4,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        learning_rate=2e-5,
        seed=42,
        eval_strategy="epoch",
        save_strategy="no",
        logging_steps=50,
        report_to="none",
        disable_tqdm=True,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=compute_metrics,
    )

    start = time.time()
    trainer.train()
    elapsed = time.time() - start

    metrics = trainer.evaluate()
    metrics["fold"] = FOLD
    metrics["train_time_sec"] = elapsed
    roberta_fold_metrics.append(metrics)
    print(f"Fold {FOLD} done: acc={metrics['eval_accuracy']:.4f} f1={metrics['eval_f1']:.4f} time={elapsed:.1f}s")

acc = [m["eval_accuracy"] for m in roberta_fold_metrics]
f1 = [m["eval_f1"] for m in roberta_fold_metrics]
times = [m["train_time_sec"] for m in roberta_fold_metrics]

print()
print(f"RoBERTa-base Mean accuracy: {np.mean(acc):.4f} (std {np.std(acc):.4f})")
print(f"RoBERTa-base Mean f1:       {np.mean(f1):.4f} (std {np.std(f1):.4f})")
print(f"RoBERTa-base Mean train time per fold: {np.mean(times):.1f}s")

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

...

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


{'loss': '0.3912', 'grad_norm': '11.97', 'learning_rate': '1.742e-05', 'epoch': '0.5263'}
{'eval_loss': '0.1243', 'eval_accuracy': '0.9643', 'eval_precision': '0.9353', 'eval_recall': '0.9641', 'eval_f1': '0.9495', 'eval_runtime': '1.978', 'eval_samples_per_second': '283.6', 'eval_steps_per_second': '18.2', 'epoch': '1'}
{'loss': '0.1427', 'grad_norm': '1.395', 'learning_rate': '1.479e-05', 'epoch': '1.053'}
{'loss': '0.05835', 'grad_norm': '88.64', 'learning_rate': '1.216e-05', 'epoch': '1.579'}
{'eval_loss': '0.1312', 'eval_accuracy': '0.9768', 'eval_precision': '0.984', 'eval_recall': '0.9487', 'eval_f1': '0.9661', 'eval_runtime': '1.988', 'eval_samples_per_second': '282.2', 'eval_steps_per_second': '18.11', 'epoch': '2'}
{'loss': '0.073', 'grad_norm': '0.01928', 'learning_rate': '9.526e-06', 'epoch': '2.105'}
{'loss': '0.02215', 'grad_norm': '18.89', 'learning_rate': '6.895e-06', 'epoch': '2.632'}
{'eval_loss': '0.1379', 'eval_accuracy': '0.9733', 'eval_precision': '0.9688', 'eval_

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


{'loss': '0.3921', 'grad_norm': '33.01', 'learning_rate': '1.773e-05', 'epoch': '0.463'}
{'loss': '0.1757', 'grad_norm': '15.31', 'learning_rate': '1.542e-05', 'epoch': '0.9259'}
{'eval_loss': '0.1004', 'eval_accuracy': '0.9688', 'eval_precision': '0.9851', 'eval_recall': '0.9614', 'eval_f1': '0.9731', 'eval_runtime': '1.277', 'eval_samples_per_second': '276.3', 'eval_steps_per_second': '18', 'epoch': '1'}
{'loss': '0.1272', 'grad_norm': '6.776', 'learning_rate': '1.31e-05', 'epoch': '1.389'}
{'loss': '0.06844', 'grad_norm': '54.64', 'learning_rate': '1.079e-05', 'epoch': '1.852'}
{'eval_loss': '0.1203', 'eval_accuracy': '0.9717', 'eval_precision': '0.9805', 'eval_recall': '0.971', 'eval_f1': '0.9757', 'eval_runtime': '1.258', 'eval_samples_per_second': '280.7', 'eval_steps_per_second': '18.29', 'epoch': '2'}
{'loss': '0.04762', 'grad_norm': '71.65', 'learning_rate': '8.472e-06', 'epoch': '2.315'}
{'loss': '0.04356', 'grad_norm': '0.1077', 'learning_rate': '6.157e-06', 'epoch': '2.778'

In [20]:
from transformers import RobertaForSequenceClassification

roberta_large_tok = AutoTokenizer.from_pretrained("roberta-large")

roberta_large_encodings = roberta_large_tok(
    list(df["text"]),
    padding="max_length",
    truncation=True,
    max_length=MAX_LENGTH,
)

roberta_large_dataset = DarkPatternDataset(roberta_large_encodings, list(df["label"]))

large_fold_metrics = []

for FOLD in range(1, 6):
    train_idx = df.index[(df["split"] == "trainval") & (df["cv_fold"] != FOLD)].tolist()
    val_idx   = df.index[(df["split"] == "trainval") & (df["cv_fold"] == FOLD)].tolist()

    train_dataset = Subset(roberta_large_dataset, train_idx)
    val_dataset   = Subset(roberta_large_dataset, val_idx)

    model = RobertaForSequenceClassification.from_pretrained("roberta-large", num_labels=2)

    training_args = TrainingArguments(
        output_dir=f"./roberta_large_fold{FOLD}_output",
        num_train_epochs=4,
        per_device_train_batch_size=8,       # halved vs base, to fit memory
        gradient_accumulation_steps=2,        # 8 x 2 = effective batch size 16, same as every other model
        per_device_eval_batch_size=16,
        learning_rate=2e-5,
        seed=42,
        eval_strategy="epoch",
        save_strategy="no",
        logging_steps=50,
        report_to="none",
        disable_tqdm=True,
        fp16=True,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=compute_metrics,
    )

    start = time.time()
    trainer.train()
    elapsed = time.time() - start

    metrics = trainer.evaluate()
    metrics["fold"] = FOLD
    metrics["train_time_sec"] = elapsed
    large_fold_metrics.append(metrics)
    print(f"Fold {FOLD} done: acc={metrics['eval_accuracy']:.4f} f1={metrics['eval_f1']:.4f} time={elapsed:.1f}s")

acc = [m["eval_accuracy"] for m in large_fold_metrics]
f1 = [m["eval_f1"] for m in large_fold_metrics]
times = [m["train_time_sec"] for m in large_fold_metrics]

print()
print(f"RoBERTa-large Mean accuracy: {np.mean(acc):.4f} (std {np.std(acc):.4f})")
print(f"RoBERTa-large Mean f1:       {np.mean(f1):.4f} (std {np.std(f1):.4f})")
print(f"RoBERTa-large Mean train time per fold: {np.mean(times):.1f}s")

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

...

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-large
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


{'loss': '0.9736', 'grad_norm': 'inf', 'learning_rate': '1.742e-05', 'epoch': '0.5291'}
{'eval_loss': '0.2648', 'eval_accuracy': '0.943', 'eval_precision': '0.8688', 'eval_recall': '0.9846', 'eval_f1': '0.9231', 'eval_runtime': '1.786', 'eval_samples_per_second': '314', 'eval_steps_per_second': '20.15', 'epoch': '1'}
{'loss': '0.3584', 'grad_norm': '0.4419', 'learning_rate': '1.479e-05', 'epoch': '1.053'}
{'loss': '0.295', 'grad_norm': '18.43', 'learning_rate': '1.216e-05', 'epoch': '1.582'}
{'eval_loss': '0.1634', 'eval_accuracy': '0.975', 'eval_precision': '0.9892', 'eval_recall': '0.9385', 'eval_f1': '0.9632', 'eval_runtime': '1.792', 'eval_samples_per_second': '313.1', 'eval_steps_per_second': '20.09', 'epoch': '2'}
{'loss': '0.2103', 'grad_norm': '0.09288', 'learning_rate': '9.526e-06', 'epoch': '2.106'}
{'loss': '0.1034', 'grad_norm': '0.764', 'learning_rate': '6.895e-06', 'epoch': '2.635'}
{'eval_loss': '0.1835', 'eval_accuracy': '0.9679', 'eval_precision': '0.9538', 'eval_recal

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-large
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


{'loss': '0.8688', 'grad_norm': '64.25', 'learning_rate': '1.773e-05', 'epoch': '0.4651'}
{'loss': '0.5086', 'grad_norm': '98.2', 'learning_rate': '1.542e-05', 'epoch': '0.9302'}
{'eval_loss': '0.26', 'eval_accuracy': '0.9462', 'eval_precision': '0.9273', 'eval_recall': '0.9855', 'eval_f1': '0.9555', 'eval_runtime': '1.161', 'eval_samples_per_second': '304.1', 'eval_steps_per_second': '19.81', 'epoch': '1'}
{'loss': '0.4732', 'grad_norm': '128.6', 'learning_rate': '1.31e-05', 'epoch': '1.391'}
{'loss': '0.168', 'grad_norm': '0.1189', 'learning_rate': '1.079e-05', 'epoch': '1.856'}
{'eval_loss': '0.2616', 'eval_accuracy': '0.9518', 'eval_precision': '0.9398', 'eval_recall': '0.9807', 'eval_f1': '0.9598', 'eval_runtime': '1.187', 'eval_samples_per_second': '297.4', 'eval_steps_per_second': '19.38', 'epoch': '2'}
{'loss': '0.1696', 'grad_norm': '0.07159', 'learning_rate': '8.472e-06', 'epoch': '2.316'}
{'loss': '0.1023', 'grad_norm': '0.01199', 'learning_rate': '6.157e-06', 'epoch': '2.78

In [21]:
import torch
import time
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix
from transformers import DistilBertForSequenceClassification, RobertaForSequenceClassification, Trainer, TrainingArguments

# ---------- SECTION 13: Error analysis via out-of-fold predictions ----------

def run_cv_with_predictions(model_class, model_name, dataset, is_large=False):
    all_preds = []
    for FOLD in range(1, 6):
        train_idx = df.index[(df["split"] == "trainval") & (df["cv_fold"] != FOLD)].tolist()
        val_idx   = df.index[(df["split"] == "trainval") & (df["cv_fold"] == FOLD)].tolist()

        train_dataset = Subset(dataset, train_idx)
        val_dataset   = Subset(dataset, val_idx)

        model = model_class.from_pretrained(model_name, num_labels=2)

        training_args = TrainingArguments(
            output_dir=f"./tmp_{model_name.replace('/', '_')}_fold{FOLD}",
            num_train_epochs=4,
            per_device_train_batch_size=8 if is_large else 16,
            gradient_accumulation_steps=2 if is_large else 1,
            per_device_eval_batch_size=16,
            learning_rate=2e-5,
            seed=42,
            eval_strategy="no",
            save_strategy="no",
            logging_steps=200,
            report_to="none",
            disable_tqdm=True,
            fp16=True,
        )

        trainer = Trainer(model=model, args=training_args, train_dataset=train_dataset)
        trainer.train()

        pred_output = trainer.predict(val_dataset)
        pred_labels = np.argmax(pred_output.predictions, axis=1)
        probs = torch.softmax(torch.tensor(pred_output.predictions), dim=1).numpy()

        fold_df = df.loc[val_idx, ["page_id", "text", "label", "Pattern Category"]].copy()
        fold_df["orig_idx"] = val_idx
        fold_df["predicted"] = pred_labels
        fold_df["confidence"] = probs.max(axis=1)
        fold_df["fold"] = FOLD
        all_preds.append(fold_df)
        print(f"  Fold {FOLD} done.")

    return pd.concat(all_preds, ignore_index=True)

print("Running DistilBERT out-of-fold predictions...")
distil_oof = run_cv_with_predictions(DistilBertForSequenceClassification, "distilbert-base-uncased", distil_dataset)

print("Running RoBERTa-large out-of-fold predictions...")
roberta_large_oof = run_cv_with_predictions(RobertaForSequenceClassification, "roberta-large", roberta_large_dataset, is_large=True)

def error_report(oof_df, name):
    errors = oof_df[oof_df.label != oof_df.predicted]
    print(f"\n=== {name} ===")
    print(f"Error rate: {len(errors)}/{len(oof_df)} = {len(errors)/len(oof_df):.2%}")
    print("Errors by Pattern Category:")
    print(errors["Pattern Category"].value_counts())
    cm = confusion_matrix(oof_df.label, oof_df.predicted)
    print("Confusion matrix [[TN, FP], [FN, TP]]:")
    print(cm)
    return errors

distil_errors = error_report(distil_oof, "DistilBERT")
roberta_errors = error_report(roberta_large_oof, "RoBERTa-large")

# Merge to see overlap: does each model fail on the SAME examples, or different ones?
merged = distil_oof[["orig_idx", "text", "label", "Pattern Category", "predicted"]].rename(columns={"predicted": "distil_pred"})
merged = merged.merge(
    roberta_large_oof[["orig_idx", "predicted"]].rename(columns={"predicted": "roberta_pred"}),
    on="orig_idx"
)
merged["distil_wrong"] = merged["label"] != merged["distil_pred"]
merged["roberta_wrong"] = merged["label"] != merged["roberta_pred"]

both_wrong = merged[merged.distil_wrong & merged.roberta_wrong]
distil_only = merged[merged.distil_wrong & ~merged.roberta_wrong]
roberta_only = merged[~merged.distil_wrong & merged.roberta_wrong]

print(f"\nBoth models wrong: {len(both_wrong)}")
print(f"DistilBERT wrong, RoBERTa-large right: {len(distil_only)}")
print(f"RoBERTa-large wrong, DistilBERT right: {len(roberta_only)}")
print("\nExamples BOTH models got wrong (genuinely hard cases):")
print(both_wrong[["text", "label", "Pattern Category"]].head(10).to_string(index=False))

# ---------- SECTION 14: Computational analysis ----------

def measure_inference(model_class, model_name, tokenizer, device):
    model = model_class.from_pretrained(model_name, num_labels=2).to(device)
    model.eval()
    sample = tokenizer("FLASH SALE | LIMITED TIME ONLY Shop Now", return_tensors="pt",
                        padding="max_length", truncation=True, max_length=MAX_LENGTH).to(device)
    with torch.no_grad():
        for _ in range(10):  # warmup
            _ = model(**sample)
    if device == "cuda":
        torch.cuda.synchronize()
    start = time.time()
    with torch.no_grad():
        for _ in range(100):
            _ = model(**sample)
    if device == "cuda":
        torch.cuda.synchronize()
    elapsed_ms = (time.time() - start) / 100 * 1000
    n_params = model.num_parameters()
    print(f"{model_name:25s} | {device:4s} | {n_params:>12,} params | {elapsed_ms:6.2f} ms/prediction")
    return n_params, elapsed_ms

print("\n=== Inference latency: single text in, one prediction out ===")
for device in ["cuda", "cpu"]:
    measure_inference(BertForSequenceClassification, "bert-base-uncased", tok, device)
    measure_inference(DistilBertForSequenceClassification, "distilbert-base-uncased", distil_tok, device)
    measure_inference(RobertaForSequenceClassification, "roberta-base", roberta_tok, device)
    measure_inference(RobertaForSequenceClassification, "roberta-large", roberta_large_tok, device)

Running DistilBERT out-of-fold predictions...


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


{'loss': '0.1862', 'grad_norm': '6.869', 'learning_rate': '1.061e-05', 'epoch': '1.887'}
{'loss': '0.04833', 'grad_norm': '0.05663', 'learning_rate': '1.179e-06', 'epoch': '3.774'}
{'train_runtime': '17.21', 'train_samples_per_second': '393.5', 'train_steps_per_second': '24.64', 'train_loss': '0.1117', 'epoch': '4'}
  Fold 1 done.


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

...

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


roberta-base              | cpu  |  124,647,170 params | 170.27 ms/prediction


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-large
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


roberta-large             | cpu  |  355,361,794 params | 998.17 ms/prediction


In [22]:
import copy
from transformers import TrainerCallback

class BestModelTracker(TrainerCallback):
    """Keeps the best-F1 epoch's weights in memory. No disk writes during training."""
    def __init__(self):
        self.best_f1 = -1
        self.best_state_dict = None
        self.best_epoch = None

    def on_evaluate(self, args, state, control, metrics, model, **kwargs):
        f1 = metrics.get("eval_f1", -1)
        if f1 > self.best_f1:
            self.best_f1 = f1
            self.best_epoch = state.epoch
            # copy weights to CPU so GPU memory isn't doubled
            self.best_state_dict = copy.deepcopy({k: v.cpu() for k, v in model.state_dict().items()})

large_fold_metrics_earlystop = []
best_epochs_used = []

for FOLD in range(1, 6):
    train_idx = df.index[(df["split"] == "trainval") & (df["cv_fold"] != FOLD)].tolist()
    val_idx   = df.index[(df["split"] == "trainval") & (df["cv_fold"] == FOLD)].tolist()

    train_dataset = Subset(roberta_large_dataset, train_idx)
    val_dataset   = Subset(roberta_large_dataset, val_idx)

    model = RobertaForSequenceClassification.from_pretrained("roberta-large", num_labels=2)
    tracker = BestModelTracker()

    training_args = TrainingArguments(
        output_dir=f"./roberta_large_es_fold{FOLD}",
        num_train_epochs=4,
        per_device_train_batch_size=8,
        gradient_accumulation_steps=2,
        per_device_eval_batch_size=16,
        learning_rate=2e-5,
        seed=42,
        eval_strategy="epoch",
        save_strategy="no",          # <-- key fix: never write checkpoints to disk
        logging_steps=50,
        report_to="none",
        disable_tqdm=True,
        fp16=True,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=compute_metrics,
        callbacks=[tracker],
    )

    trainer.train()

    # reload the best epoch's weights (from memory, instant) before final scoring
    model.load_state_dict(tracker.best_state_dict)
    metrics = trainer.evaluate()
    metrics["fold"] = FOLD
    large_fold_metrics_earlystop.append(metrics)
    best_epochs_used.append(tracker.best_epoch)

    print(f"Fold {FOLD}: acc={metrics['eval_accuracy']:.4f} f1={metrics['eval_f1']:.4f} best_epoch={tracker.best_epoch}")

acc = [m["eval_accuracy"] for m in large_fold_metrics_earlystop]
f1 = [m["eval_f1"] for m in large_fold_metrics_earlystop]

print()
print(f"Early-stopped RoBERTa-large Mean accuracy: {np.mean(acc):.4f} (std {np.std(acc):.4f})")
print(f"Early-stopped RoBERTa-large Mean f1:       {np.mean(f1):.4f} (std {np.std(f1):.4f})")
print("Best epoch per fold:", best_epochs_used)
print()
print("Compare to fixed-4-epoch result: accuracy=0.9746, f1=0.9736")

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-large
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


{'loss': '0.6533', 'grad_norm': '10.84', 'learning_rate': '1.769e-05', 'epoch': '0.4717'}
{'loss': '0.3948', 'grad_norm': '6.786', 'learning_rate': '1.533e-05', 'epoch': '0.9434'}
{'eval_loss': '0.1385', 'eval_accuracy': '0.9603', 'eval_precision': '0.9843', 'eval_recall': '0.94', 'eval_f1': '0.9616', 'eval_runtime': '1.167', 'eval_samples_per_second': '323.8', 'eval_steps_per_second': '20.56', 'epoch': '1'}
{'loss': '0.2889', 'grad_norm': '52.99', 'learning_rate': '1.297e-05', 'epoch': '1.415'}
{'loss': '0.2276', 'grad_norm': '172.5', 'learning_rate': '1.061e-05', 'epoch': '1.887'}
{'eval_loss': '0.122', 'eval_accuracy': '0.9788', 'eval_precision': '0.9848', 'eval_recall': '0.975', 'eval_f1': '0.9799', 'eval_runtime': '1.177', 'eval_samples_per_second': '321.1', 'eval_steps_per_second': '20.39', 'epoch': '2'}
{'loss': '0.1344', 'grad_norm': '0.5272', 'learning_rate': '8.255e-06', 'epoch': '2.358'}
{'loss': '0.06059', 'grad_norm': '1.898', 'learning_rate': '5.896e-06', 'epoch': '2.83'}

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-large
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-large
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


{'loss': '0.9736', 'grad_norm': 'inf', 'learning_rate': '1.742e-05', 'epoch': '0.5291'}
{'eval_loss': '0.2648', 'eval_accuracy': '0.943', 'eval_precision': '0.8688', 'eval_recall': '0.9846', 'eval_f1': '0.9231', 'eval_runtime': '1.792', 'eval_samples_per_second': '313.1', 'eval_steps_per_second': '20.09', 'epoch': '1'}
{'loss': '0.3584', 'grad_norm': '0.4419', 'learning_rate': '1.479e-05', 'epoch': '1.053'}
{'loss': '0.295', 'grad_norm': '18.43', 'learning_rate': '1.216e-05', 'epoch': '1.582'}
{'eval_loss': '0.1634', 'eval_accuracy': '0.975', 'eval_precision': '0.9892', 'eval_recall': '0.9385', 'eval_f1': '0.9632', 'eval_runtime': '1.772', 'eval_samples_per_second': '316.6', 'eval_steps_per_second': '20.32', 'epoch': '2'}
{'loss': '0.2103', 'grad_norm': '0.09288', 'learning_rate': '9.526e-06', 'epoch': '2.106'}
{'loss': '0.1034', 'grad_norm': '0.764', 'learning_rate': '6.895e-06', 'epoch': '2.635'}
{'eval_loss': '0.1835', 'eval_accuracy': '0.9679', 'eval_precision': '0.9538', 'eval_rec

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-large
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


{'loss': '0.8688', 'grad_norm': '64.25', 'learning_rate': '1.773e-05', 'epoch': '0.4651'}
{'loss': '0.5086', 'grad_norm': '98.2', 'learning_rate': '1.542e-05', 'epoch': '0.9302'}
{'eval_loss': '0.26', 'eval_accuracy': '0.9462', 'eval_precision': '0.9273', 'eval_recall': '0.9855', 'eval_f1': '0.9555', 'eval_runtime': '1.154', 'eval_samples_per_second': '305.8', 'eval_steps_per_second': '19.92', 'epoch': '1'}
{'loss': '0.4732', 'grad_norm': '128.6', 'learning_rate': '1.31e-05', 'epoch': '1.391'}
{'loss': '0.168', 'grad_norm': '0.1189', 'learning_rate': '1.079e-05', 'epoch': '1.856'}
{'eval_loss': '0.2616', 'eval_accuracy': '0.9518', 'eval_precision': '0.9398', 'eval_recall': '0.9807', 'eval_f1': '0.9598', 'eval_runtime': '1.173', 'eval_samples_per_second': '300.9', 'eval_steps_per_second': '19.6', 'epoch': '2'}
{'loss': '0.1696', 'grad_norm': '0.07159', 'learning_rate': '8.472e-06', 'epoch': '2.316'}
{'loss': '0.1023', 'grad_norm': '0.01199', 'learning_rate': '6.157e-06', 'epoch': '2.781

In [23]:
train_full_idx = df.index[df["split"] == "trainval"].tolist()
test_idx = df.index[df["split"] == "test"].tolist()

train_full_dataset = Subset(roberta_large_dataset, train_full_idx)
test_dataset = Subset(roberta_large_dataset, test_idx)

print(f"Training on: {len(train_full_dataset)} rows (the full CV pool)")
print(f"Evaluating once on: {len(test_dataset)} rows (never touched before now)")

model = RobertaForSequenceClassification.from_pretrained("roberta-large", num_labels=2)

training_args = TrainingArguments(
    output_dir="./roberta_large_final_eval",
    num_train_epochs=4,              # matches the locked-in configuration, no tuning
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    seed=42,
    eval_strategy="no",              # no mid-training evaluation - we're not selecting anything here
    save_strategy="no",
    logging_steps=50,
    report_to="none",
    disable_tqdm=True,
    fp16=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_full_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

trainer.train()
final_test_metrics = trainer.evaluate()
print()
print("=== FINAL HELD-OUT TEST SET RESULT (report this number) ===")
print(final_test_metrics)

Training on: 2071 rows (the full CV pool)
Evaluating once on: 284 rows (never touched before now)


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-large
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


{'loss': '0.8564', 'grad_norm': '91.43', 'learning_rate': '1.812e-05', 'epoch': '0.3861'}
{'loss': '0.4013', 'grad_norm': '104.1', 'learning_rate': '1.619e-05', 'epoch': '0.7722'}
{'loss': '0.2538', 'grad_norm': '22.13', 'learning_rate': '1.427e-05', 'epoch': '1.154'}
{'loss': '0.2403', 'grad_norm': '0.02011', 'learning_rate': '1.235e-05', 'epoch': '1.541'}
{'loss': '0.1474', 'grad_norm': '1.235', 'learning_rate': '1.042e-05', 'epoch': '1.927'}
{'loss': '0.09727', 'grad_norm': '0.05132', 'learning_rate': '8.5e-06', 'epoch': '2.309'}
{'loss': '0.07582', 'grad_norm': '2.856', 'learning_rate': '6.577e-06', 'epoch': '2.695'}
{'loss': '0.1245', 'grad_norm': '0.02096', 'learning_rate': '4.654e-06', 'epoch': '3.077'}
{'loss': '0.05528', 'grad_norm': '0.01203', 'learning_rate': '2.731e-06', 'epoch': '3.463'}
{'loss': '0.03335', 'grad_norm': '0.00748', 'learning_rate': '8.077e-07', 'epoch': '3.849'}
{'train_runtime': '139.9', 'train_samples_per_second': '59.21', 'train_steps_per_second': '3.717

In [24]:
full_data_idx = df.index.tolist()  # every row - train+val+test combined
full_training_dataset = Subset(roberta_large_dataset, full_data_idx)

print(f"Training final deployment model on: {len(full_training_dataset)} rows (100% of data)")

final_model = RobertaForSequenceClassification.from_pretrained("roberta-large", num_labels=2)

final_training_args = TrainingArguments(
    output_dir="./roberta_large_deployment",
    num_train_epochs=4,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=2e-5,
    seed=42,
    eval_strategy="no",     # no evaluation - nothing left to hold out, this is the final artifact
    save_strategy="no",
    logging_steps=50,
    report_to="none",
    disable_tqdm=True,
    fp16=True,
)

final_trainer = Trainer(
    model=final_model,
    args=final_training_args,
    train_dataset=full_training_dataset,
)

final_trainer.train()

# Save model + tokenizer together - the backend team needs both
final_model.save_pretrained("./darkguard_deployment_model")
roberta_large_tok.save_pretrained("./darkguard_deployment_model")

import shutil
shutil.make_archive("darkguard_roberta_large_final", "zip", "./darkguard_deployment_model")

print()
print("Saved and zipped: darkguard_roberta_large_final.zip")


Training final deployment model on: 2355 rows (100% of data)


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-large
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


{'loss': '0.8369', 'grad_norm': '9.382', 'learning_rate': '1.834e-05', 'epoch': '0.339'}
{'loss': '0.2615', 'grad_norm': '4.947', 'learning_rate': '1.666e-05', 'epoch': '0.678'}
{'loss': '0.3421', 'grad_norm': '0.06651', 'learning_rate': '1.497e-05', 'epoch': '1.014'}
{'loss': '0.2527', 'grad_norm': '0.3527', 'learning_rate': '1.328e-05', 'epoch': '1.353'}
{'loss': '0.1327', 'grad_norm': '33.64', 'learning_rate': '1.159e-05', 'epoch': '1.692'}
{'loss': '0.2448', 'grad_norm': '0.01478', 'learning_rate': '9.899e-06', 'epoch': '2.027'}
{'loss': '0.1034', 'grad_norm': '0.05786', 'learning_rate': '8.209e-06', 'epoch': '2.366'}
{'loss': '0.06376', 'grad_norm': '0.04142', 'learning_rate': '6.52e-06', 'epoch': '2.705'}
{'loss': '0.0654', 'grad_norm': '0.02332', 'learning_rate': '4.831e-06', 'epoch': '3.041'}
{'loss': '0.003424', 'grad_norm': '0.02746', 'learning_rate': '3.142e-06', 'epoch': '3.38'}
{'loss': '0.03218', 'grad_norm': '0.6671', 'learning_rate': '1.453e-06', 'epoch': '3.719'}
{'tra

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Saved and zipped: darkguard_roberta_large_final.zip
